# Budget vs Actual Ingestion (Azure Cost Management Budgets API)

Pulls **budgets** defined in Azure Cost Management (`Microsoft.Consumption/budgets`) for the
same scope(s) used in `Cost_API_Ingestion_Sample.ipynb`, and lands them alongside their
**current spend** and **forecast spend** so the dashboard can show **budget vs actual**
per capacity/scope, and flag overspend before month-end.

Uses the same auth pattern as the other notebooks in this folder —
`notebookutils.credentials.getToken` for the identity running the notebook.

**Required access**: **Cost Management Reader** (to read budgets) at each scope. Creating
budgets still requires **Cost Management Contributor** and is out of scope for this notebook
— define budgets once in the Azure Portal (or via IaC), this notebook only reads them.

> ⚠️ Azure budgets are commonly scoped to a whole subscription/resource group, not to
> "Fabric spend" specifically. For a clean budget-vs-actual comparison against
> `cost_fabric_api` (which is already filtered to `ServiceName = 'Microsoft Fabric'`),
> create the budget with a **filter** on `ServiceName = Microsoft Fabric` (or scope it to a
> resource group dedicated to Fabric capacities) when you set it up in Cost Management —
> otherwise `CurrentSpendAmount` here will include non-Fabric spend too.

## Step 0 – Parameters

In [ ]:
from notebookutils import mssparkutils
import requests, json
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

# Same scopes you're already pulling cost for
scopes = [
    "/subscriptions/00000000-0000-0000-0000-000000000000"
]

api_version = "2024-08-01"
management_endpoint = "https://management.azure.com"
target_table = "budgets"

In [ ]:
import time

def get_arm_token(max_attempts=3, base_delay_seconds=5):
    """
    See the Cost notebook's identical helper for context: retries a
    transient 500 INTERNAL_ERROR from Fabric's token broker before
    surfacing the error. If it still fails, restart the notebook session.
    """
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return notebookutils.credentials.getToken("https://management.azure.com/")
        except Exception as e:
            last_error = e
            print(f"WARN: getToken attempt {attempt}/{max_attempts} failed: {e}")
            if attempt < max_attempts:
                time.sleep(base_delay_seconds * attempt)
    raise RuntimeError(
        "Failed to acquire an ARM token after retries. Restart the notebook session "
        "(Stop session, then run again) and retry -- see Cost_API_Ingestion_Sample.ipynb "
        "for the fuller troubleshooting note."
    ) from last_error

arm_token = get_arm_token()
headers = {
    "Authorization": f"Bearer {arm_token}",
    "Content-Type": "application/json"
}

## Step 1 – List budgets for each scope

In [ ]:
def list_budgets(scope, headers):
    url = f"{management_endpoint}{scope}/providers/Microsoft.Consumption/budgets?api-version={api_version}"
    results = []
    while url:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 429:
            import time
            time.sleep(int(resp.headers.get("Retry-After", "10")))
            continue
        resp.raise_for_status()
        payload = resp.json()
        results.extend(payload.get("value", []))
        url = payload.get("nextLink")
    return results

all_budgets = []
for scope in scopes:
    print(f"Listing budgets for scope: {scope}")
    for b in list_budgets(scope, headers):
        props = b.get("properties", {})
        current_spend = props.get("currentSpend") or {}
        forecast_spend = props.get("forecastSpend") or {}
        time_period = props.get("timePeriod") or {}
        all_budgets.append(Row(
            Scope=scope,
            BudgetName=b.get("name"),
            Category=props.get("category"),
            Amount=float(props.get("amount", 0) or 0),
            TimeGrain=props.get("timeGrain"),
            StartDate=time_period.get("startDate"),
            EndDate=time_period.get("endDate"),
            CurrentSpendAmount=float(current_spend.get("amount", 0) or 0),
            CurrentSpendUnit=current_spend.get("unit"),
            ForecastSpendAmount=float(forecast_spend.get("amount", 0) or 0) if forecast_spend else None,
            ForecastSpendUnit=forecast_spend.get("unit") if forecast_spend else None,
        ))

print(f"Retrieved {len(all_budgets)} budget(s) across {len(scopes)} scope(s)")

## Step 2 – Compute budget-vs-actual variance and land into a Delta table

In [ ]:
if all_budgets:
    df = spark.createDataFrame(all_budgets)
    df = (
        df
        .withColumn("VarianceAmount", df.CurrentSpendAmount - df.Amount)
        .withColumn("VariancePct", (df.CurrentSpendAmount - df.Amount) / df.Amount * 100)
        .withColumn("ForecastVsBudgetPct", (df.ForecastSpendAmount / df.Amount) * 100)
        .withColumn("IngestedAt", current_timestamp())
    )

    # Append with a timestamp so budget-vs-actual can be trended over time,
    # not just viewed as a single current snapshot.
    df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(target_table)

    over_budget = df.filter(df.VarianceAmount > 0).count()
    print(f"Wrote {df.count()} row(s) to '{target_table}' — {over_budget} budget(s) currently over amount")
    display(df)
else:
    print("No budgets found for the given scope(s) — create one in Cost Management first.")

## Next steps

- Schedule this notebook alongside the cost/utilization ones — budgets refresh their
  `currentSpend` roughly daily in Cost Management, so a daily run is enough.
- In the report, plot `CurrentSpendAmount` against `Amount` per budget/scope over time (from
  the appended `IngestedAt` history) — a line crossing the budget threshold is the "cost
  went up" alert the utilization notebook's `IsThrottled` flag can help explain.
- If you don't want to depend on someone having created Azure budgets, an alternative is a
  small manually-maintained "target" Delta table (`CapacityId, Month, TargetAmount`) that you
  compare `fact_cost_fabric` against directly — happy to add that as a fallback if the Azure
  budgets route doesn't fit your setup.